# Mây AI Studio trên Google Colab
Trò chuyện · Tạo ảnh · Dựng video bằng mô hình mã nguồn mở, chạy trên GPU miễn phí của Colab.

**Trước tiên:** menu *Runtime → Change runtime type → T4 GPU → Save*. Sau đó chạy lần lượt từng ô (bấm ▶).

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Lấy mã nguồn
**Cách dễ nhất:** kéo file `ai-studio.zip` vào khung **Files** (biểu tượng thư mục bên trái), đợi tải xong rồi chạy ô dưới.
Nếu repo GitHub của bạn để công khai thì không cần tải zip, ô dưới tự lấy từ GitHub.

In [ ]:
import os
%cd /content
if os.path.exists('ai-studio.zip'):
    !unzip -oq ai-studio.zip
elif not os.path.exists('ai-studio'):
    !git clone -q --depth 1 -b ccr-b6e744b3-y0noaa https://github.com/thanhdangkhoa16122008-hash/TDK tdk && mv tdk/ai-studio ai-studio
assert os.path.exists('ai-studio/app.py'), 'Chưa có mã nguồn: hãy kéo ai-studio.zip vào khung Files rồi chạy lại ô này.'
%cd /content/ai-studio
print('✓ Đã có mã nguồn')

In [ ]:
# 3. Cài thư viện (~2–3 phút)
!pip install -q -r requirements.txt

## 4. Lưu vào Google Drive (rất nên chạy)
Colab xóa mọi thứ khi tắt. Ô này giữ lại: mô hình đã tải (~30 GB, khỏi tải lại), LoRA đã huấn luyện, và **trí nhớ + tài liệu + dữ liệu học của Mây**: thứ làm Mây trở thành AI của riêng bạn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/ai-studio'
for sub in ['loras', 'userdata', 'hf-cache']:
    os.makedirs(f'{D}/{sub}', exist_ok=True)
os.environ['HF_HOME'] = f'{D}/hf-cache'          # mô hình tải 1 lần
os.environ['STUDIO_HOME'] = f'{D}/userdata'     # trí nhớ, tài liệu, dữ liệu học của Mây
!rm -rf loras && ln -s $D/loras loras
print('✓ Mô hình, LoRA và trí nhớ của Mây sẽ được giữ trong Google Drive')

## 5. Chạy AI Studio
Đợi dòng `Running on public URL: https://….gradio.live` rồi bấm vào link. Lần đầu mỗi tính năng sẽ tải mô hình (chat ~6 GB, ảnh ~7 GB, video ~10–20 GB) nên chờ vài phút.

In [ ]:
!python app.py --share

---
# Huấn luyện thêm (tùy chọn)
Dừng ô 5 (■) trước khi huấn luyện để giải phóng GPU.

### A. Dạy Mây cách nói chuyện của bạn (LoRA chat)
Sửa/thêm hội thoại vào `data/chat_vi.jsonl` (mỗi dòng một hội thoại). Càng nhiều hội thoại chất lượng (300–1000) càng tốt. T4: ~10–40 phút.

In [ ]:
# dùng cả dữ liệu mẫu lẫn các câu bạn đã 👍 trong lúc chat
!python train/train_chat_lora.py --data data/chat_vi.jsonl $STUDIO_HOME/feedback_good.jsonl --epochs 2

### B. Dạy phong cách ảnh / nhân vật riêng (LoRA ảnh)
Tải 15–40 ảnh vào `data/images/muro-var/` (khung Files bên trái), rồi chạy. T4: ~30–60 phút cho 800 bước.

In [ ]:
!python train/train_image_lora.py --images data/images/muro-var --name muro-var --prompt "murovar style, black shadow figure" --steps 800

Sau khi huấn luyện, chạy lại ô 5. LoRA chat tự được dùng; LoRA ảnh chọn trong tab **Tạo ảnh → Nâng cao**, và nhớ viết từ khóa (vd `murovar style`) trong mô tả.